# 3-bit repetition codes

This notebook demonstrates the repetition-code compiler passes. `rep3_bit` encodes each logical qubit as three physical qubits and decodes measurements with `qstack.decode @majority_vote`. `rep3_phase` is the phase-flip sibling: allocated logical zero is prepared as `|+++>`, measurements are taken in the phase basis, and results are decoded with `qstack.decode @phase_majority_vote`.

Because both passes are module-to-module transformations over the same qstack IR, composing them builds the 9-qubit Shor-style demonstration code.


## 1. A tiny logical program

We prepare $|1\rangle$ by applying `X` to $|0\rangle$ and measure it. In
the unprotected program one `X` gate is enough — but if any gate is
followed by a bit-flip, the measurement is wrong with probability 1.

In [ ]:
%load_ext qstack.jupyter

In [ ]:
%%qasm logical
QSTACKQASM 0.1;
include "qstack/cliffords.inc";

qreg q[1];
creg c[1];
x q[0];
measure q[0] -> c[0];

In [ ]:
print(logical)

## 2. Compile with the 3-bit repetition pass

`compile_rep3_bit` rewrites the module:

- every `!qstack.qubit` value becomes **three** physical qubits,
- every single-qubit Clifford (`H`, `X`, `Z`, `S`) is applied
  transversally on each of the three copies,
- every `qstack.measure` becomes three measurements followed by a
  `qstack.decode @majority_vote` outside the kernel boundary.

Compare the printed IR to the logical version above.

In [ ]:
from qstack.passes.rep3_bit import compile_rep3_bit, register_rep3_bit_callbacks

encoded = compile_rep3_bit(logical)
print(encoded)

## 3. Run the encoded program

The encoded module still surfaces a single classical bit through `main`.
We register the `majority_vote` decoder on a `CallbackRegistry` and run
1000 shots — every shot should return `[1]`.

In [ ]:
from qstack.runtime import CallbackRegistry, Machine

reg = CallbackRegistry()
register_rep3_bit_callbacks(reg)

machine = Machine(encoded, num_qubits=4, registry=reg)
results = machine.eval(shots=1000)
results

In [ ]:
results.plot_histogram()

## 4. Compose bit and phase repetition into a Shor-style code

Applying the bit pass and then the phase pass widens each logical qubit to nine physical qubits. The original program and callback boundary do not change; the module simply gains another QEC layer and another decoder declaration.


In [ ]:
from qstack.passes.rep3_phase import compile_rep3_phase, register_rep3_phase_callbacks

shor = compile_rep3_phase(encoded)
register_rep3_phase_callbacks(reg)
print(shor)


Notice the structure in the printed IR:

- the kernel now allocates **9** physical qubits,
- each bit-code physical lane is expanded by the phase-code pass,
- `qstack.decode @phase_majority_vote` collapses each outer phase triple,
- the original `qstack.decode @majority_vote` remains as the inner bit-code decoder.

Running the composed program only requires adding the phase decoder to the same callback registry.


In [ ]:
machine2 = Machine(shor, num_qubits=9, registry=reg)
results2 = machine2.eval(shots=1000)
results2.plot_histogram()


Two different repetition passes, one callback registry, same `Machine`: this is the Shor-code composition pattern. The compiler output remains regular qstack IR, so it can still feed later hardware-lowering passes.
